## Notes on MATLAB implementation

When setting up an elastic network model in matlab, this is what I do:

https://github.com/ando-lab/mdx-examples/blob/main/lys_tet_model/job03_goodvibes_model.m

```matlab
ENT = proc.script.ElasticNetworkTools.initialize(Atoms,Basis,SpaceGroup);
[g,Tg] = ENT.find_atom_groups();
```

The `ElasticNetworkTools` is doing this:

https://github.com/ando-lab/mdx-lib/blob/57a845447cff519a0037390997b2799b7e81361f/%2Bproc/%2Bscript/ElasticNetworkTools.m#L278-L281

```matlab
Atoms = Atoms(~Atoms.isHet & Atoms.mdxAtomicSymbol~="H",:); % remove waters and hydrogens
ENT = proc.script.ElasticNetworkTools('Atoms',Atoms,'Basis',Basis);
ENT.tlsorigin = ENT.com;
ENT.UnitCellOperators = ENT.map_operators_to_cell(SpaceGroup.generalPositions);
```

The member function `find_atom_groups` does this:

```matlab
A = obj.Atoms(:,obj.groupAttributes);
[A,sortorder] = sortrows(A);
[~,unsortorder] = sort(sortorder);
[group_assignments,Tg] = findgroups(A);
group_assignments = group_assignments(unsortorder);

```

This is similar to groupby in pandas. The default value for groupAttributes is: {'mdxGroupByResidue'}. The mdxGroupByResidue column is assigned by `proc.script.ImportPDB().run(opts.pdbFileName)` which calls `assign_residue_category` with the following:

```matlab
% now, group each heteroatom with its the nearest residue
res = zeros(size(A,1),1);
isIncl = A.mdxResidueCategory == 'protein' | A.mdxResidueCategory == 'nucleic acid';
[~,~,res(isIncl)] = unique(A(isIncl,{'chainID','resSeq'}),'rows');
res = proc.script.ImportPDB.assign_by_proximity(A,res);
A.mdxGroupByResidue = res;
```

I think I can ignore the resiodue group thing for now, the important thing to get right is the symmetry expansion of the enm (the `externalModel` -- those contacts between an atom in the asu and a symmetry related atom). This is done by `ENT.externalModel`:

```matlab
T = obj.externalContactSearch();
[T,G,index,Tg] = obj.contacts2nodes(T);
```

The contact search calls `proc.script.CoordinateTools.symmetry_neighbor_search`. 

...

In [1]:
from goodvibes import enm
import gemmi
import pandas as pd

In [2]:
#coordinate_file = 'test_data/2OLX.cif'
#coordinate_file = 'test_data/6O2H.cif'
coordinate_file = 'test_data/lys_1_refmac.pdb' 

st = gemmi.read_structure(coordinate_file)
st.setup_entities()  # supposed to be good practice
# drop ligands and waters:
st.remove_ligands_and_waters()
enm._pack_unit_cell(st, inplace=True) # pack the unit cell (also drops ligands and waters by default)
df = enm._find_contacts(
    st, 
    distance_cutoff=4.0, 
    include_h=False,
    ) # find contacts within the unit cell

In [3]:
# some stats:
is_internal = (df["sym_idx1"] == df["sym_idx2"]) & (df["pbc_shift1"] == df["pbc_shift2"])
is_intra_unit_cell = df["pbc_shift1"] == df["pbc_shift2"]
print('before symmetry expansion, there are this many contacts:')
print('  within ASUs:',df[is_internal].shape[0])
print('  between ASUs:',df[~is_internal].shape[0])
print('  between unit cells:',df[~is_intra_unit_cell].shape[0])
print('  between ASUs within the same unit cell:',df[is_intra_unit_cell & ~is_internal].shape[0])

before symmetry expansion, there are this many contacts:
  within ASUs: 6056
  between ASUs: 202
  between unit cells: 163
  between ASUs within the same unit cell: 39


In [4]:
df2 = enm._symmetry_expand(df, st.cell.images)

In [5]:
# some stats, after symmetry expansion:
is_internal = (df2["sym_idx1"] == df2["sym_idx2"]) & (df2["pbc_shift1"] == df2["pbc_shift2"])
is_intra_unit_cell = df2["pbc_shift1"] == df2["pbc_shift2"]
print('after symmetry expansion, there are this many contacts:')
print('  within ASUs:',df2[is_internal].shape[0])
print('  between ASUs:',df2[~is_internal].shape[0])
print('  between unit cells:',df2[~is_intra_unit_cell].shape[0])
print('  between ASUs within the same unit cell:',df2[is_intra_unit_cell & ~is_internal].shape[0])

after symmetry expansion, there are this many contacts:
  within ASUs: 48448
  between ASUs: 1305
  between unit cells: 994
  between ASUs within the same unit cell: 311


In [6]:
# verify that none of the rows has pbc_shift1 != (0,0,0)
assert (df2['pbc_shift1'] != (0,0,0)).sum() == 0

In [7]:
# compare with MATLAB output

import pandas as pd

df_ml = pd.read_csv('test_data/lys_1_enm_edges.csv')

def group_cols(df, cols, name):
    df[name] = df[cols].apply(tuple, axis=1)
    df.drop(columns=cols,inplace=True)
    return df

group_cols(df_ml,['c2_1', 'c2_2', 'c2_3'],'c2')
group_cols(df_ml,['r1_1', 'r1_2', 'r1_3'],'r1')
group_cols(df_ml,['r2_1', 'r2_2', 'r2_3'],'r2')

# how many rows?
print('MATLAB output has this many contacts:', df_ml.shape[0])

MATLAB output has this many contacts: 1305


In [8]:
# what are the unique values of o1, o2, interface?
print('MATLAB output has these unique values of o1:', df_ml['o1'].unique())
print('MATLAB output has these unique values of o2:', df_ml['o2'].unique())
print('MATLAB output has these unique values of interface:', df_ml['interface'].unique())

MATLAB output has these unique values of o1: [1 2 3 4 5 6 7 8]
MATLAB output has these unique values of o2: [3 5 7 8 1 2 4 6]
MATLAB output has these unique values of interface: [ 1  2 -2 -1  3  4  5]


In [9]:
# lets just look at the rows with o1==1
# summarize the number of occurances of each value of o2:
print('MATLAB output has these counts of o2 values for rows with o1==1:')
print(df_ml[df_ml['o1']==1]['o2'].value_counts())

MATLAB output has these counts of o2 values for rows with o1==1:
o2
8    69
3    54
5    54
7    25
Name: count, dtype: int64


In [10]:
# do the same for python version before symmetry expansion (df).
# here, o1 and o2 are zero-indexed, and the column names are
# 'sym_idx1', 'sym_idx2'
print('Python output has these counts of sym_idx2 values for rows with sym_idx1==0:')
print(df[df['sym_idx1']==0]['sym_idx2'].value_counts())



Python output has these counts of sym_idx2 values for rows with sym_idx1==0:
sym_idx2
0    6056
7      69
3      54
1      54
5      25
Name: count, dtype: int64


In [11]:
# to gain some insight, let's compare 
# rows with o1==1 and o2==7 in matlab output vs.
# rows with sym_idx1==0 and sym_idx2==5 in python output
# just print the tables:

# for matlab, only print: a1, a2, g1, g2, c2 
print('MATLAB output rows with o1==1 and o2==7:')
df_ml[(df_ml['o1']==1) & (df_ml['o2']==7)][[ 'a1', 'a2', 'g1', 'g2', 'c2']]

MATLAB output rows with o1==1 and o2==7:


,a1,a2,g1,g2,c2
108,1,5,10,14,"(0, 0, -1)"
109,2,98,13,129,"(0, 0, -1)"
110,2,99,13,129,"(0, 0, -1)"
111,3,98,13,129,"(0, 0, -1)"
112,3,99,13,129,"(0, 0, -1)"
113,4,94,13,128,"(0, 0, -1)"
114,5,1,14,10,"(0, 0, -1)"
115,6,94,14,128,"(0, 0, -1)"
116,6,97,14,128,"(0, 0, -1)"
117,7,94,16,128,"(0, 0, -1)"


In [12]:
df[(df['sym_idx1']==0) & (df['sym_idx2']==5)][['cra1','cra2','pbc_shift2']]

,cra1,cra2,pbc_shift2
486,A/ALA 10/CB,A/ARG 14/CD,"(0, 0, -1)"
617,A/LYS 13/CE,A/LEU 129/O,"(0, 0, -1)"
619,A/LYS 13/CE,A/LEU 129/C,"(0, 0, -1)"
621,A/LYS 13/NZ,A/LEU 129/O,"(0, 0, -1)"
622,A/LYS 13/NZ,A/LEU 129/C,"(0, 0, -1)"
636,A/LYS 13/O,A/ARG 128/NE,"(0, 0, -1)"
665,A/ARG 14/CD,A/ALA 10/CB,"(0, 0, -1)"
683,A/ARG 14/O,A/ARG 128/NH2,"(0, 0, -1)"
686,A/ARG 14/O,A/ARG 128/NE,"(0, 0, -1)"
739,A/GLY 16/N,A/ARG 128/NE,"(0, 0, -1)"
